In [1]:
import os
import sys
import math
import copy 
import torch 
import random
DEBUG = False
import numpy as np
import torch_sparse
import pandas as pd 
import pickle as pkl
from tqdm import tqdm
from time import time
import networkx as nx
from dgl import DGLGraph
from scipy import linalg
from pathlib import Path
from torch import Tensor
import scipy.sparse as sp
from random import randint
from dgl import transforms
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from scipy import sparse, stats
from scipy.sparse import csgraph
from scipy.sparse import csr_matrix
from dgl import from_networkx, DGLGraph
from torch_geometric.utils import scatter
from dgl.data import citation_graph as citegrh
from torch_geometric.typing import SparseTensor
from torch_geometric.utils import to_undirected
from torch_geometric.utils import remove_self_loops
from sklearn.metrics.pairwise import cosine_similarity
# from ipynb.fs.full.Dataset import get_data_from_dataset
# from ipynb.fs.full.Dataset import get_data_from_dataset,train_val_test_mask
from ogb.nodeproppred import Evaluator, PygNodePropPredDataset
from typing import Callable, List, NamedTuple, Optional, Tuple, Union
from torch_geometric.utils import add_self_loops,add_remaining_self_loops
# from ipynb.fs.full.SpectralSparsifier import EffectiveResistance, LocalEffectiveResistance, get_sparse_adj_matrix

In [3]:
import numpy as np
import torch
import pickle as pkl
import sys
import networkx as nx
import numpy as np
import scipy.sparse as sp

import argparse
import time
from time import localtime
import torch
import torch.nn.functional as F
from dgl import DGLGraph
from dgl.data import register_data_args, load_data
import random
from torch.backends import cudnn
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import dgl.function as fn
from dgl.nn.pytorch import edge_softmax
import numpy as np
import torch.nn.functional as F

import torch
import torch.nn as nn
import dgl.function as fn
from dgl.nn.pytorch import edge_softmax
import numpy as np
import torch.nn.functional as F


In [4]:
import DeviceDir

DIR, RESULTS_DIR = DeviceDir.get_directory()
device, NUM_PROCESSORS = DeviceDir.get_device()

In [61]:
from ipynb.fs.full.SGSLoadDataset import LOAD_DATASET

DATASET_NAME = "Squirrel"
data, dataset  = LOAD_DATASET(DIR, DATASET_NAME)
num_classes = max(data.y).item()+1

Squirrel N: 5201 E: 396846 F: 2345 C: 5 d: 76.30 lr: 0.48 i: False s: True u: True


In [62]:
sig = nn.Sigmoid()
hardtanh = nn.Hardtanh(0,1)
gamma = -0.1
zeta = 1.1
beta = 0.66
eps = 1e-20
const1 = beta*np.log(-gamma/zeta + eps)

def l0_train(logAlpha, min, max):
    U = torch.rand(logAlpha.size()).type_as(logAlpha) + eps
    s = sig((torch.log(U / (1 - U)) + logAlpha) / beta)
    s_bar = s * (zeta - gamma) + gamma
    mask = F.hardtanh(s_bar, min, max)
    return mask

def l0_test(logAlpha, min, max):
    s = sig(logAlpha/beta)
    s_bar = s * (zeta - gamma) + gamma
    mask = F.hardtanh(s_bar, min, max)
    return mask

def get_loss2(logAlpha):
    return sig(logAlpha - const1)


class GraphAttention(nn.Module):
    def __init__(self,
                 g,
                 in_dim,
                 out_dim,
                 num_heads,
                 feat_drop,
                 attn_drop,
                 alpha,
                 bias_l0,
                 residual=False,l0=0, min=0):
        super(GraphAttention, self).__init__()
        self.g = g
        self.num_heads = num_heads
        self.fc = nn.Linear(in_dim, num_heads * out_dim, bias=False)
        if feat_drop:
            self.feat_drop = nn.Dropout(feat_drop)
        else:
            self.feat_drop = lambda x : x
        if attn_drop:
            self.attn_drop = nn.Dropout(attn_drop)
        else:
            self.attn_drop = lambda x : x
        self.attn_l = nn.Parameter(torch.Tensor(size=(1, 1, out_dim)))
        self.attn_r = nn.Parameter(torch.Tensor(size=(1, 1, out_dim)))
        self.bias_l0 = nn.Parameter(torch.FloatTensor([bias_l0]))

        nn.init.xavier_normal_(self.fc.weight.data, gain=1.414)
        nn.init.xavier_normal_(self.attn_l.data, gain=1.414)
        nn.init.xavier_normal_(self.attn_r.data, gain=1.414)
        self.leaky_relu = nn.LeakyReLU(alpha)
        self.softmax = edge_softmax
        self.residual = residual
        self.num = 0
        self.l0 = l0
        self.loss = 0
        self.dis = []
        self.min=min
        if residual:
            if in_dim != out_dim:
                self.res_fc = nn.Linear(in_dim, num_heads * out_dim, bias=False)
                nn.init.xavier_normal_(self.res_fc.weight.data, gain=1.414)
            else:
                self.res_fc = None

    def forward(self, inputs, edges="__ALL__", skip=0):
        self.loss = 0
        # prepare
        h = self.feat_drop(inputs)  # NxD
        ft = self.fc(h).reshape((h.shape[0], self.num_heads, -1))  # NxHxD'
        a1 = (ft * self.attn_l).sum(dim=-1).unsqueeze(-1) # N x H x 1
        a2 = (ft * self.attn_r).sum(dim=-1).unsqueeze(-1) # N x H x 1
      
        self.g.ndata.update({'ft' : ft, 'a1' : a1, 'a2' : a2})

        if skip == 0:
            # 1. compute edge attention
            self.g.apply_edges(self.edge_attention, edges)

            # 2. compute softmax
            if self.l0 == 1:
                ind = self.g.nodes()
                self.g.apply_edges(self.loop, edges=(ind, ind))

            self.edge_softmax()

            if self.l0 == 1:
                self.g.apply_edges(self.norm)

        # 2. compute the aggregated node features scaled by the dropped,
            edges = self.g.edata['a'].squeeze().nonzero().squeeze()


        self.g.edata['a_drop'] = self.attn_drop(self.g.edata['a'])
        self.num = (self.g.edata['a'] > 0).sum()
        self.g.update_all(fn.u_mul_e('ft', 'a_drop', 'ft'), fn.sum('ft', 'ft'))
        ret = self.g.ndata['ft']

        # 4. residual
        if self.residual:
            if self.res_fc is not None:
                resval = self.res_fc(h).reshape((h.shape[0], self.num_heads, -1))  # NxHxD'
            else:
                resval = torch.unsqueeze(h, 1)  # Nx1xD'
            ret = resval + ret
        return ret, edges

    def edge_attention(self, edges):
        # an edge UDF to compute unnormalized attention values from src and dst
        if self.l0 == 0:
            m = self.leaky_relu(edges.src['a1'] + edges.dst['a2'])
        else:
            tmp = edges.src['a1'] + edges.dst['a2']
            logits = tmp + self.bias_l0

            if self.training:
                m = l0_train(logits, 0, 1)
            else:
                m = l0_test(logits, 0, 1)
            self.loss = get_loss2(logits[:,0,:]).sum()
        return {'a': m}

    def norm(self, edges):
        # normalize attention
        a = edges.data['a'] / edges.dst['z']
        return {'a' : a}

    def loop(self, edges):
        # set attention to itself as 1
        return {'a': torch.pow(edges.data['a'], 0)}

    def normalize(self, logits):
        self._logits_name = "_logits"
        self._normalizer_name = "_norm"
        self.g.edata[self._logits_name] = logits
        self.g.update_all(fn.copy_edge(self._logits_name, self._logits_name),
                         fn.sum(self._logits_name, self._normalizer_name))
        return self.g.edata.pop(self._logits_name), self.g.ndata.pop(self._normalizer_name)

    def edge_softmax(self):

        if self.l0 == 0:
            scores = self.softmax(self.g, self.g.edata.pop('a'))
        else:
            scores, normalizer = self.normalize(self.g.edata.pop('a'))
            self.g.ndata['z'] = normalizer[:,0,:].unsqueeze(1)

        self.g.edata['a'] = scores[:,0,:].unsqueeze(1)

class GAT(nn.Module):
    def __init__(self,
                 g,
                 num_layers,
                 in_dim,
                 num_hidden,
                 num_classes,
                 heads,
                 activation,
                 feat_drop,
                 attn_drop,
                 alpha,
                 bias_l0,
                 residual, l0=0):
        super(GAT, self).__init__()
        self.g = g
        self.num_layers = num_layers
        self.gat_layers = nn.ModuleList()
        self.activation = activation
        # input projection (no residual)
        self.gat_layers.append(GraphAttention(
            g, in_dim, num_hidden, heads[0], feat_drop, attn_drop, alpha,bias_l0, False, l0=l0, min=0))
        # hidden layers
        for l in range(1, num_layers):
            # due to multi-head, the in_dim = num_hidden * num_heads
            self.gat_layers.append(GraphAttention(
                g, num_hidden * heads[l-1], num_hidden, heads[l],
                feat_drop, attn_drop, alpha,bias_l0, residual, l0=l0, min=0))
        # output projection
        self.gat_layers.append(GraphAttention(
            g, num_hidden * heads[-2], num_classes, heads[-1],
            feat_drop, attn_drop, alpha,bias_l0, residual, l0=l0))


    def forward(self, inputs):

        h = inputs
        edges = "__ALL__"
        h, edges = self.gat_layers[0](h, edges)
        h = self.activation(h.flatten(1))
        for l in range(1, self.num_layers):
            h, _= self.gat_layers[l](h, edges, skip=1)
            h = self.activation(h.flatten(1))

        # output projection
        logits,_ = self.gat_layers[-1](h, edges, skip=1)
        logits = logits.mean(1)
        return logits

utils .py

In [63]:
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def step(self, acc, model):
        score = acc
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score:
            self.counter += 1
            # print('EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0
        return self.early_stop

    def save_checkpoint(self, model):
        '''Saves model when validation loss decrease.'''
        torch.save(model.state_dict(), 'es_checkpoint.pt')

GAT . py

In [64]:
from sklearn.metrics import f1_score

def accuracy(logits, labels):
#     _, indices = torch.max(logits, dim=1)
#     correct = torch.sum(indices == labels)
#     return correct.item() * 1.0 / len(labels)
    preds = logits.argmax(dim=1)
    f1 = f1_score(labels.cpu(), preds.cpu(), average='micro')
    return f1
    

def evaluate(model, features, labels, mask,loss_fcn):
    model.eval()
    with torch.no_grad():
        logits = model(features)
        logits = logits[mask]
        labels = labels[mask]
        loss_data = loss_fcn(logits, labels)
        return accuracy(logits, labels), loss_data

def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if gpu >= 0:
        torch.cuda.manual_seed(seed)
        cudnn.benchmark = False
        cudnn.deterministic = True

In [79]:
from ipynb.fs.full.SGSLoadDataset import LOAD_DATASET

DATASET_NAME = "Roman-empire"
data, dataset  = LOAD_DATASET(DIR, DATASET_NAME)
num_classes = max(data.y).item()+1

Roman-empire N: 22662 E: 65854 F: 300 C: 18 d: 2.91 lr: 0.50 i: False s: False u: True


In [86]:
num_out_heads = 4
num_heads = 16
num_layers = 1
gpu = -1
l0 = 0
num_hidden = 128
residual = False 
idrop = 0.6
adrop = 0.6
lr = 0.005 
weight_decay = 5e-4
alpha = 0.2
early_stop = True 
fastmode = False 
seed = 123 
bias = 0 
loss_lo = 0 
syn_type = "scipy" 
self_loop = False 
sess ='default'
loss_l0  = 0
weight_decay = 0
l0 = 0

#data, dataset = get_data_from_dataset('cornell5')
epochs = 500

data = data.to(device)

#features = torch.FloatTensor(data.x)
features = data.x
#labels = torch.LongTensor(data.y)
labels = data.y
train_mask = data.train_mask
val_mask = data.val_mask
test_mask = data.test_mask
num_feats = features.shape[1]
n_classes = max(data.y).item()+1
n_edges = data.num_edges
current_time = time.strftime('%d_%H:%M:%S', localtime())

print("""----Data statistics------'
    #Edges %d
    #Classes %d
    #Train samples %d
    #Val samples %d
    #Test samples %d""" %
        (n_edges, n_classes,
        train_mask.sum().item(),
        val_mask.sum().item(),
        test_mask.sum().item()))

cuda = True

import dgl
from dgl.data.utils import generate_mask_tensor

g = dgl.graph((data.edge_index[0], data.edge_index[1]), num_nodes=data.x.size(0))

# Add node features
g.ndata['feat'] = data.x
g.ndata['label'] = data.y
# g.ndata['train_mask'] = generate_mask_tensor(data.train_mask.numpy())
# g.ndata['val_mask'] = generate_mask_tensor(data.val_mask.numpy())
# g.ndata['test_mask'] = generate_mask_tensor(data.test_mask.numpy())

g.ndata['train_mask'] = data.train_mask
g.ndata['val_mask'] = data.val_mask
g.ndata['test_mask'] = data.test_mask


n_edges = g.number_of_edges()
heads = ([num_heads] * num_layers) + [num_out_heads]

model = GAT(g,
            num_layers,
            num_feats,
            num_hidden,
            n_classes,
            heads,
            F.elu,
            idrop,
            adrop,
            alpha,
            bias,
            residual,
            l0).to(device)
print(model)

if early_stop:
    stopper = EarlyStopping(patience=150)
loss_fcn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
dur = []
time_used = 0

----Data statistics------'
    #Edges 65854
    #Classes 18
    #Train samples 11331
    #Val samples 5665
    #Test samples 5666
GAT(
  (gat_layers): ModuleList(
    (0): GraphAttention(
      (fc): Linear(in_features=300, out_features=2048, bias=False)
      (feat_drop): Dropout(p=0.6, inplace=False)
      (attn_drop): Dropout(p=0.6, inplace=False)
      (leaky_relu): LeakyReLU(negative_slope=0.2)
    )
    (1): GraphAttention(
      (fc): Linear(in_features=2048, out_features=72, bias=False)
      (feat_drop): Dropout(p=0.6, inplace=False)
      (attn_drop): Dropout(p=0.6, inplace=False)
      (leaky_relu): LeakyReLU(negative_slope=0.2)
    )
  )
)


In [87]:
dur = []
best_test_f1 = 0
last_5_losses = []
convergence_threshold = 0.001  # Define your convergence threshold

for epoch in range(epochs):
    model.train()
    t0 = time.time()

    # Forward pass
    logits = model(features)
    loss = loss_fcn(logits[train_mask], labels[train_mask])
    loss_l0 = loss_l0 * model.gat_layers[0].loss

    # Backward pass and optimization
    optimizer.zero_grad()
    (loss + loss_l0).backward()
    optimizer.step()
    dur.append(time.time() - t0)

    train_acc = accuracy(logits[train_mask], labels[train_mask])

    # Validation accuracy
    if fastmode:
        val_acc, loss = accuracy(logits[val_mask], labels[val_mask], loss_fcn)
    else:
        val_acc, _ = evaluate(model, features, labels, val_mask, loss_fcn)

    # Track last 5 losses
    last_5_losses.append(loss.item())
    if len(last_5_losses) > 5:
        last_5_losses.pop(0)

    # Check convergence
    if epoch >= 5 and np.std(last_5_losses) < convergence_threshold:
        print(f"Convergence achieved at Epoch: {epoch:05d} | Std of Last 5 Losses: {np.std(last_5_losses):.4f}")
        break

    print("Epoch {:05d} | Time(s) {:.4f} | Loss {:.4f} | TrainAcc {:.4f} |"
          " ValAcc {:.4f} | ETputs(KTEPS) {:.2f}".format(epoch, np.mean(dur), loss.item(), train_acc,
                                                         val_acc, n_edges / np.mean(dur) / 1000))

print(dataset.name)
print("Time per epoch: ", np.mean(dur))

acc, _ = evaluate(model,features, labels, test_mask, loss_fcn)
print("Test Accuracy {:.4f}".format(acc))


Epoch 00000 | Time(s) 0.0152 | Loss 2.8910 | TrainAcc 0.0498 | ValAcc 0.1901 | ETputs(KTEPS) 4334.30
Epoch 00001 | Time(s) 0.0138 | Loss 2.7336 | TrainAcc 0.1364 | ValAcc 0.2302 | ETputs(KTEPS) 4765.60
Epoch 00002 | Time(s) 0.0135 | Loss 2.7136 | TrainAcc 0.1333 | ValAcc 0.3124 | ETputs(KTEPS) 4893.70
Epoch 00003 | Time(s) 0.0132 | Loss 2.6668 | TrainAcc 0.1549 | ValAcc 0.3435 | ETputs(KTEPS) 5002.45
Epoch 00004 | Time(s) 0.0131 | Loss 2.6404 | TrainAcc 0.1638 | ValAcc 0.3398 | ETputs(KTEPS) 5035.07
Epoch 00005 | Time(s) 0.0130 | Loss 2.6498 | TrainAcc 0.1634 | ValAcc 0.3772 | ETputs(KTEPS) 5054.59
Epoch 00006 | Time(s) 0.0130 | Loss 2.6258 | TrainAcc 0.1687 | ValAcc 0.3714 | ETputs(KTEPS) 5070.08
Epoch 00007 | Time(s) 0.0130 | Loss 2.6332 | TrainAcc 0.1627 | ValAcc 0.3626 | ETputs(KTEPS) 5082.66
Epoch 00008 | Time(s) 0.0129 | Loss 2.6224 | TrainAcc 0.1664 | ValAcc 0.3520 | ETputs(KTEPS) 5087.81
Epoch 00009 | Time(s) 0.0129 | Loss 2.6162 | TrainAcc 0.1693 | ValAcc 0.3576 | ETputs(KTEPS

Epoch 00084 | Time(s) 0.0127 | Loss 2.5647 | TrainAcc 0.1766 | ValAcc 0.4203 | ETputs(KTEPS) 5200.52
Epoch 00085 | Time(s) 0.0127 | Loss 2.5793 | TrainAcc 0.1741 | ValAcc 0.4256 | ETputs(KTEPS) 5200.12
Epoch 00086 | Time(s) 0.0127 | Loss 2.5700 | TrainAcc 0.1758 | ValAcc 0.4208 | ETputs(KTEPS) 5200.09
Epoch 00087 | Time(s) 0.0127 | Loss 2.5704 | TrainAcc 0.1768 | ValAcc 0.4076 | ETputs(KTEPS) 5199.62
Epoch 00088 | Time(s) 0.0127 | Loss 2.5755 | TrainAcc 0.1726 | ValAcc 0.4034 | ETputs(KTEPS) 5198.90
Epoch 00089 | Time(s) 0.0127 | Loss 2.5680 | TrainAcc 0.1772 | ValAcc 0.4007 | ETputs(KTEPS) 5198.83
Epoch 00090 | Time(s) 0.0127 | Loss 2.5682 | TrainAcc 0.1724 | ValAcc 0.4067 | ETputs(KTEPS) 5198.59
Epoch 00091 | Time(s) 0.0127 | Loss 2.5699 | TrainAcc 0.1706 | ValAcc 0.4180 | ETputs(KTEPS) 5198.21
Epoch 00092 | Time(s) 0.0127 | Loss 2.5776 | TrainAcc 0.1758 | ValAcc 0.4221 | ETputs(KTEPS) 5198.10
Epoch 00093 | Time(s) 0.0127 | Loss 2.5842 | TrainAcc 0.1773 | ValAcc 0.4229 | ETputs(KTEPS

Test Model Accuracy

Calculating Mean and Standard Deviation

In [78]:
import numpy as np

# Define the list
a = [0.4155,0.4238,0.4179]

# Calculate mean and standard deviation
mean_a = np.mean(a)
std_a = np.std(a)

# Print the result formatted to 4 decimal places
print(f"Mean: {mean_a:.4f} +/- {std_a:.4f}")

Mean: 0.4100 +/- 0.0089
